# SMBHB Population Poisson Example

This notebook reproduces the Poisson-sampled SMBHB workflow using the `fastropop` package API rather than hardcoded local functions.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import fastropop
from fastropop import SemiAnalyticPopulation, binning


In [ ]:
population_params = {
    "n0": 10**-90.4153,
    "alphaM": -1.3800,
    "Mstar": 10**8.8272 * fastropop.MsunMKS,
    "betaz": -0.1711,
    "z0": 4.70,
}

integration_limits = {
    "fbounds": [fastropop.fminNG15 * fastropop.s, 10**-7.53],
}

pop = SemiAnalyticPopulation(
    population_params=population_params,
    integration_limits=integration_limits,
)


In [ ]:
M_vals = np.logspace(6, 11, 500)
y_vals = np.array([pop.d2ndzdM(1.0, M * fastropop.MsunMKS) for M in M_vals])

plt.figure(figsize=(7, 5))
plt.loglog(M_vals, y_vals, color="C1")
plt.xlabel(r"$M\ [M_\odot]$")
plt.ylabel(r"$d^2 n/(dz d\mathcal{M})$")
plt.title(r"Mass Function at $z = 1$")
plt.grid(True, which="both", ls="--", alpha=0.6)
plt.show()


In [ ]:
M_vals = np.logspace(6, 10, 100) * fastropop.MsunMKS
dndlogM_vals = np.array([pop.dndlog10M(M) for M in M_vals])

plt.figure(figsize=(7, 5))
plt.loglog(M_vals / fastropop.MsunMKS, dndlogM_vals)
plt.xlabel(r"$M/M_\odot$")
plt.ylabel(r"$dn/d\log_{10}(M/M_\odot)$")
plt.ylim(10**-9, 10**3)
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.show()


In [ ]:
freqs = 10 ** np.arange(-9, -7, 0.2)
hc_values = np.array([np.sqrt(pop.hc2(f)) for f in freqs])

plt.figure(figsize=(7, 5))
plt.semilogx(freqs, np.log10(hc_values), marker="o", ls="-", color="C0", label="Numerical")
plt.semilogx(freqs, -(np.log10(freqs[0]**(-2/3)) - np.log10(hc_values[0])) + np.log10(freqs**(-2/3)), color="C1", ls="--", label=r"$f^{-2/3}$")
plt.xlabel(r"$f$ [Hz]")
plt.ylabel(r"$\log_{10}(h_c)$")
plt.title(r"Characteristic Strain Spectrum $h_c(f)$")
plt.legend()
plt.grid(True, which="both", ls="--", alpha=0.6)
plt.show()


In [ ]:
Nbinaries_mean = pop.compute_Nbinaries(verbose=True)
Nbinaries_draw = int(pop.generate_poisson_realization(1, key=0)[0])
Nbinaries_mean, Nbinaries_draw


In [ ]:
distM, distz, distlog10f = pop.sample_dist(Nbinaries=Nbinaries_draw, key=1)
spec = binning(distM, distz, distlog10f, do_plot=False)

plt.figure(figsize=(7, 5))
plt.plot(np.asarray(spec[:, 0]), np.log10(np.sqrt(np.asarray(spec[:, 1]))), marker="o", ls="-", lw=1.8, ms=4, label="Poisson sampled")
plt.plot(np.log10(freqs), np.log10(hc_values), "-", color="C2", label="Ref")
plt.xlabel(r"$f$ [Hz]")
plt.ylabel(r"$h_c(f)$")
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.title("Single realization")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
tabreal, log10f, median, q_low, q_high = pop.compute_many_realizations(
    Nbinaries=Nbinaries_mean,
    nrealizations=50,
    freqs=freqs,
    hc2_values=hc_values,
    do_plot=True,
    key=2,
)
